In [ ]:
import requests
from bs4 import BeautifulSoup
import time, random, json, pandas as pd, re, pickle
from tqdm import tqdm
from datetime import datetime  # ✅ Thay pd.Timestamp

# Rotating User Agents
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:125.0) Gecko/20100101 Firefox/125.0',
]

# Config brands
BRANDS_CONFIG = {
   #"samsung": 6
   # "huawei": 6
   #"xiaomi": 8,
    #"oppo": 6,
    "honor": 4,
    "realme": 4,
    "vivo": 7,
    # "tecno": 3,
    # "google": 1,
    # "zte": 3,
    # "oneplus": 2,
     # "sony": 1,
     # "nothing": 1,
     # "infinix": 1,
     # "meizu": 1
}

BASE_URL = "https://www.gsmarena.com/"
WANTED_SECTIONS = {"Body", "Display", "Platform", "Memory", "Main Camera", 
                  "Selfie camera", "Sound", "Comms", "Features", "Battery"}

def get_random_headers():
    """Rotating headers"""
    return {
        'User-Agent': random.choice(USER_AGENTS),
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Connection': 'keep-alive',
    }

class AntiBanSession:
    def __init__(self):
        self.session_counter = 0
        self.session = None
        
    def get_session(self):
        if self.session_counter % 30 == 0 or self.session is None:
            print("🔄 NEW SESSION + Fresh UA")
            self.session = requests.Session()
            self.session.headers.update(get_random_headers())
            time.sleep(random.uniform(45, 90))
            self.session_counter = 0
        self.session_counter += 1
        return self.session

class MultiBrandScraper:
    def __init__(self):
        self.anti_ban = AntiBanSession()
        self.progress_file = "samsung_progress.pkl"
        
    def safe_get(self, url: str, retries=6):
        """✅ safe_get ĐÚNG với self"""
        session = self.anti_ban.get_session()
        
        for attempt in range(retries):
            session.headers.update(get_random_headers())
            
            try:
                print(f"📡 {url.split('/')[-1]}")
                resp = session.get(url, timeout=25)
                
                if resp.status_code == 429:
                    wait = 120 + random.randint(30, 90)
                    print(f"🚫 429 → Chờ {wait}s")
                    time.sleep(wait)
                    continue
                    
                if resp.status_code == 200:
                    return BeautifulSoup(resp.content, "html.parser")
                else:
                    print(f"❌ HTTP {resp.status_code}")
                    
            except Exception as e:
                print(f"❌ Error: {e}")
                time.sleep(45)
        return None
    
    def scrape_specs(self, slug: str) -> dict:
        """✅ scrape_specs ĐÚNG"""
        url = BASE_URL + slug
        soup = self.safe_get(url)
        if not soup:
            return {}
        
        title = soup.find("h1", class_="specs-phone-name-title")
        specs = {
            "name": title.get_text(strip=True) if title else slug,
            "gsmarena_url": url,
        }
        
        for table in soup.select("#specs-list table"):
            th = table.find("th")
            if not th or th.get_text(strip=True) not in WANTED_SECTIONS:
                continue
            section = th.get_text(strip=True)
            for row in table.select("tr"):
                key_td = row.find("td", class_="ttl")
                val_td = row.find("td", class_="nfo")
                if key_td and val_td:
                    key = key_td.get_text(strip=True)
                    val = re.sub(r"[\r\n\t]+", " ", val_td.get_text(" ", strip=True)).strip()
                    specs[f"{section} | {key}"] = val
        return specs
    
    def get_brand_id_and_prefix(self, brand_name: str) -> tuple:
        BRAND_INFO = {
            "samsung": ("9", "samsung"),
            "huawei": ("58", "huawei"),
            "xiaomi": ("80", "xiaomi"),
            "oppo": ("82", "oppo"),
            "tecno": ("120", "tecno"), 
            "google": ("107", "google"),
            "zte": ("62", "zte"),
            "oneplus": ("95", "oneplus"),
            "sony": ("7", "sony"),
            "nothing": ("128", "nothing"),
            "infinix": ("119", "infinix"),
            "meizu": ("74", "meizu"),
            "honor": ("121", "honor"),
            "vivo": ("98", "vivo"),
            "realme": ("118", "realme"),
            
        }
        return BRAND_INFO.get(brand_name.lower(), (None, None))
    
    def fetch_brand_phones(self, brand_name: str, max_pages: int) -> list:
        brand_id, brand_prefix = self.get_brand_id_and_prefix(brand_name)
        if not brand_id:
            print(f"❌ Brand không tồn tại: {brand_name}")
            return []
        
        phones = []
        page = 1
        
        print(f"\n📱 [{brand_name.upper()}] {max_pages} pages...")
        
        while page <= max_pages:
            if page == 1:
                url = f"{BASE_URL}{brand_prefix}-phones-{brand_id}.php"
            else:
                url = f"{BASE_URL}{brand_prefix}-f-{brand_id}-0-p{page}.php"
            
            print(f"  📄 Trang {page}/{max_pages}")
            soup = self.safe_get(url)
            
            if not soup:
                break
            
            items = soup.select("div.makers ul li")
            if not items:
                break
            
            new_phones = 0
            for li in items:
                a = li.find("a", href=True)
                if a:
                    phones.append({
                        "brand": brand_name,
                        "name": a.get_text(strip=True),
                        "slug": a["href"]
                    })
                    new_phones += 1
            
            print(f"  ✅ +{new_phones} (tổng: {len(phones)})")
            page += 1
            time.sleep(random.uniform(4, 7))
        
        return phones
    
    def scrape_all_brands(self):
        all_phones = []
        for brand_name, max_pages in BRANDS_CONFIG.items():
            print(f"\n{'='*60}")
            print(f"🚀 {brand_name.upper()} ({max_pages} pages)")
            print(f"{'='*60}")
            
            brand_phones = self.fetch_brand_phones(brand_name, max_pages)
            all_phones.extend(brand_phones)
            time.sleep(120)  # 2p giữa brands
        
        print(f"\n📊 TỔNG: {len(all_phones)} phones")
        return all_phones

def main():
    scraper = MultiBrandScraper()
    
    # 1. Lấy danh sách Samsung (6 pages)
    phones = scraper.scrape_all_brands()
    total_phones = len(phones)
    
    print(f"\n📱 Samsung: {total_phones} phones")
    
    # 2. Scrape specs 50-phone strategy
    print("\n🔬 SPECS SCRAPING (50 phones → 5p break)")
    all_specs = []
    
    for i, phone in enumerate(tqdm(phones, desc="Samsung Phones")):
        print(f"\n[{i+1:4d}/{total_phones}] {phone['name'][:50]}...")
        
        specs = scraper.scrape_specs(phone["slug"])
        if specs:
            specs.update({"brand": "samsung", "model": phone['name']})
            all_specs.append(specs)
        
        # ✅ 50 PHONES + 5P BREAK
        if (i + 1) % 50 == 0:
            print(f"\n🎉 50 COMPLETE! 😴 5 phút break...")
            time.sleep(300)
        else:
            time.sleep(random.uniform(10, 16))
    
    # 3. Save
    df = pd.DataFrame(all_specs)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    filename = f"samsung_{len(df)}_phones_{timestamp}.csv"
    
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    
    print(f"\n✅ DONE! {len(df)} Samsung specs")
    print(f"💾 {filename}")
    print("\n📋 Top 10:")
    print(df[['model', 'gsmarena_url']].head(10).to_string(index=False))

if __name__ == "__main__":
    main()